# Re-extract landmarks at HIGHER temporal resolution (T_native=64)
Same MediaPipe pipeline as the original cache, only the temporal length changed:
`max_frames 64 -> 128`, `T_native 32 -> 64`. Output goes to a NEW dir so the T=32
cache is untouched. ~5-6 h CPU (resume markers survive disconnects). GPU off.

**Attach the raw frames dataset** `gaurs86/wita-full-english-122signers`. Internet ON.
Fast alternative: set MAX_FRAMES=64 below for a ~3 h run (same time as the original,
keeps all 64 detected frames instead of downsampling to 32 — still 2x model resolution).


## Cell 1 — install + clone (MediaPipe pinned to match the original cache)


In [ ]:
%%capture
!pip install editdistance scipy --quiet
!pip install 'mediapipe==0.10.14' --quiet
import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')
import mediapipe


In [ ]:
print('mediapipe', mediapipe.__version__, '(must be 0.10.14)')


## Cell 2 — locate the 6 raw source dirs


In [ ]:
import os, glob
hits = glob.glob('/kaggle/input/**/eng_train_lex', recursive=True)
assert hits, 'attach gaurs86/wita-full-english-122signers (eng_train_lex not found)'
DATASET_ROOT = os.path.dirname(hits[0]); print('DATASET_ROOT =', DATASET_ROOT)
PATH_TO_LABEL = {}
for split in ('train','val','test'):
    for subset in ('lex','nonlex'):
        outer = os.path.join(DATASET_ROOT, f'eng_{split}_{subset}')
        if os.path.isdir(outer):
            PATH_TO_LABEL[outer] = (split, subset)
            n = len(glob.glob(os.path.join(outer,'**','gt.txt'), recursive=True))
            print(f'  {os.path.basename(outer):<22} -> {split}/{subset:<7} ({n} gt.txt)')
assert len(PATH_TO_LABEL)==6, f'found {len(PATH_TO_LABEL)}/6 source dirs'


## Cell 3 — output dir + resume markers + the temporal knobs


In [ ]:
MAX_FRAMES = 128     # raw frames read per clip (was 64).  64 = fast ~3h option.
T_NATIVE   = 64      # model timesteps per clip (was 32).  <-- the temporal length
OUT_ROOT   = '/kaggle/working/landmark_cache_122_t64'
MARKER_DIR = os.path.join(OUT_ROOT, '_markers'); os.makedirs(MARKER_DIR, exist_ok=True)
def marker_path(split, subset): return os.path.join(MARKER_DIR, f'{split}_{subset}_DONE')
SPLIT_ORDER = {'val':0,'test':1,'train':2}   # small splits first (fail fast)
source_paths = sorted(PATH_TO_LABEL, key=lambda p:(SPLIT_ORDER[PATH_TO_LABEL[p][0]], PATH_TO_LABEL[p][1]))
print(f'OUT_ROOT={OUT_ROOT}  MAX_FRAMES={MAX_FRAMES}  T_NATIVE={T_NATIVE}')


## Cell 4 — extract (4-process pool; resume-aware; ~5-6 h at MAX_FRAMES=128)


In [ ]:
from wita_v2.datasets.landmark_cache_122 import extract_dir_per_clip_landmarks_parallel
import json
all_stats = {}
for p in source_paths:
    split, subset = PATH_TO_LABEL[p]
    mk = marker_path(split, subset)
    if os.path.exists(mk):
        print(f'[skip] {split}/{subset} already done'); continue
    print(f'\n>>> {split}/{subset}  from {os.path.basename(p)}  (max_frames={MAX_FRAMES}, T_native={T_NATIVE})')
    stats = extract_dir_per_clip_landmarks_parallel(
        dir_path=p, out_dir=OUT_ROOT, split=split, subset=subset,
        n_workers=4, max_frames=MAX_FRAMES, T_native=T_NATIVE, overwrite=False)
    all_stats[f'{split}_{subset}'] = stats
    json.dump(stats, open(mk,'w'), indent=2, default=str)
print('\nall sources processed')


## Cell 5 — sanity: shape must be (64, 190)


In [ ]:
import numpy as np, random
from pathlib import Path
all_npz = list(Path(OUT_ROOT).rglob('*.npz'))
print('total npz:', len(all_npz)); assert all_npz, 'nothing extracted'
random.seed(42); sample = random.sample(all_npz, min(100, len(all_npz)))
feats = np.stack([np.load(p, allow_pickle=False)['feature'].astype(np.float32) for p in sample])
print('shape', feats.shape[1:], '| mean %.4f std %.4f min %.3f max %.3f' % (feats.mean(), feats.std(), feats.min(), feats.max()))
assert feats.shape[1:] == (T_NATIVE, 190), feats.shape[1:]
assert np.all(np.isfinite(feats)), 'non-finite values'
for split in ('train','val','test'):
    for subset in ('lex','nonlex'):
        n = len(list((Path(OUT_ROOT)/split/subset).glob('*.npz')))
        print(f'  {split}/{subset:<7}: {n}')


## Cell 6 — save it
**Save Version -> Save & Run All.** Then: notebook **Output -> New Dataset**, name it
**`wita-full-english-landmark-cache-t64`**. Attach that to the retrain notebook
(`run_stage18_aug_kaggle.ipynb`): set `CACHE_ROOT = _find_dir('landmark_cache_122_t64')`
and `P_AFFINE = 0.0` to isolate the temporal effect vs Stage 11's 0.4498.
